In [ ]:
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path.cwd()))

# Problem 1: SO101 Analytical IK (10 pts)

**Trivia.** 
SO-101 arm is our favorite open source robot manipulator. 
We will be working with it a lot, so better get used to it. 
One of the best ways to become familiar with all the dance moves this cutie has is to compute forward and inverse kinematics by hand, draw some diagrams and remember some trigonometry. 
Same-level tabletop manipulation is the most used setup for the SO-101 in practice (because it is natural and simple). 
For tasks such as pick-and-place, traditional solutions oftentimes rely on the notion of **pre-grasp** pose which is derived from the object we have to manipulate. 
So it is only natural to ask ourselves the question, how do we position our robot, i.e. which angles should robot joints be having, for the end-effector to attain the pre-grasp pose. 
In this task we are to answer this question with various tools.

<video src="assets/so101_cube_pickup.mp4" autoplay muted loop playsinline controls style="max-width: 480px;"></video>

**Model.** 
For simplicity we restrict our pre-grasp poses to the ones where the gripper approaches from above with z-axis pointing straight down in the world frame, and the in-plane rotation (yaw about world Z) is aligned with the object (cube) orientation. This reduces the task to **position (x, y, z) + yaw**, i.e. 4 DOF. One could argue that for simple box pick-up there is $\text{yaw} \sim \text{yaw} + \pi$ symmetry in gripper orientation as there is no difference whether we grab with moving finger from left or from the right. But this symmetry breaks when we attach camera or grab some non-symmetric object.

In the class we played around with the SO101 arm and computed forward kinematics and inverse kinematics using numerical solvers implemented in `pytorch_kinematics` library.

<img src="assets/so101_geometry_diagram.png" alt="SO101 forward kinematics" style="max-width: 640px;" />

Note that for safety (namely wiring) the wrist and gripper links have stoppers that limit range of continuous rotation to below $2\pi$. Thus they are not perfectly aligned in the home position.

<video src="assets/extra_angle_offset_explanation.mp4" autoplay muted loop playsinline controls style="max-width: 480px;"></video>

**Task.**
Implement two functions in `solutions/so101_ik.py`:
- `numerical_ik_so101_downturned`: build the 4×4 target transform for the desired (position, yaw), then solve using `pk.PseudoInverseIK`. Return the best converged joint vector. Use exact data from `robot.urdf` to compute the target transform.
- `so101_downturned_ik_symbolic`: return a dict mapping each of the five joint names (see `SO101_JOINT_NAMES`) to a sympy expression in terms of `(x, y, z, yaw)`. Use the simplified model from the image above to derive the formulas. A helper `analytical_ik_so101_downturned` lambdifies your formulas for evaluation.

Hidden tests compare your sympy expressions against the reference. Verify with the visualization below and the unit tests. Why and how analytical solution differs from the numerical one? Think about how the exact and numerical solutions behave when self-collisions or unreachable poses are considered.

Below is an interactive visualization of SO101 forward kinematics. Note the approximate "solvable" region. It does not account for robot self-collisions and does not account for solvability in the $z<0$ region.

In [ ]:
from lib.so101_ik import show_so101_viewer

show_so101_viewer()

Some examples of poses your algorithm should find:

<img src="assets/so101_pre_grasp_grid.png" alt="Examples of pre-grasp poses" style="max-width: 640px;" />

Assess your IK visually:

In [ ]:
from pathlib import Path
from lib.so101_ik import show_pre_grasp_grid
from solutions.so101_ik import numerical_ik_so101_downturned

SO101_URDF = Path("assets/so101/robot.urdf")
PRE_GRASP_POSES = [
    (np.array([0.2297, 0.0938, 0.0177]), -0.3468),
    (np.array([0.1495, -0.0688, 0.0385]), -0.0191),
    (np.array([0.0182, 0.0688, 0.0010]), 0.3199),
    (np.array([0.2516, 0.0063, 0.0094]), 0.1926),
    (np.array([0.0255, -0.2063, 0.0594]), 0.5860),
    (np.array([0.0109, -0.2063, 0.0135]), 0.0843),
    (np.array([0.0693, -0.1938, 0.0448]), 0.4232),
    (np.array([0.0255, -0.2188, 0.0385]), -0.1398),
    (np.array([0.0839, 0.2313, 0.0156]), -0.2442),
    (np.array([0.2151, -0.0563, 0.0552]), -0.3217),
]
show_pre_grasp_grid(SO101_URDF, PRE_GRASP_POSES, ik_solver=numerical_ik_so101_downturned)

# Problem 2: Broom Racing (15 pts to Gryffindor)

**Trivia.** 
You are a brave Gryffindor wizard trying out for the Quidditch team.
Currently you are tied with an arrogant Slytherin apprentice for the place in a team, so the committee has set a series of tests to decide who is the better broom handler. 
Magical cruise control keeps your forward speed constant, but riding a broom has limits: you cannot turn too sharply (centrifugal force would throw you off), and you cannot pitch the broom too far up or down (you would slip and fall). 
There are three contests: (1) fly through a Quidditch gate – a ring high in the air, so narrow that only one passing direction is safe; (2) catch the Snitch at a given point in space; (3) catch the ball and carry it through the gate (combination of the first two). 
In each duel you both start at some identical position and orientation in the air and must finish the task faster than your opponent. 
Use your LLMos spell and [pyrseltongue](https://harrypotter.fandom.com/wiki/Parseltongue) to estimate the shortest path. 
You should be equal or better than your opponent on any of the challenges.

**Model.** 
With $v$ as speed (we use units so $v = 1$), $\theta$ the heading angle, $\phi$ the pitch angle, and $u_1$, $u_2$ the control inputs (left–right and up–down steering), the equations of motion are
$$
\begin{cases} 
\dot{x} = v \cos \theta \cos \phi \\
\dot{y} = v \sin \theta \cos \phi \\
\dot{z} = v \sin \phi \\
\dot{\theta} = \frac{u_1}{\cos \phi} \\
\dot{\phi} = u_2 \\
\end{cases}
$$
Safety constraints: 
* pitch is limited by $\phi \in [\phi_{\min}, \phi_{\max}]$, and 
* path curvature is bounded by $\kappa = \sqrt{u_1^2 + u_2^2} \leq \kappa_{\max}$.

For simplicity take $\phi_{\min} = -\phi_{\max} = -45^{\circ}$ and $\kappa_{\max} = 1$.

Task requires you to output $[0, 1]$-parametrized curve $r(s)$ that satisfies equation and constraints listed above under the reparametrization $t(s) = \int_0^s \|\dot{r(\tau)}\| d\tau$ (in other words, $r(0)$ is the starting pose, $r(1)$ is the final pose and everything in between is uniform along the curve's length).

Your opponent tries to follow a heuristic length-optimal path described in [Dubins3d, Vana](https://comrob.fel.cvut.cz/papers/icra20dubins3d.pdf). For tasks where direction of visited configuration is not specified it is chosen by some heuristic rule. It is guaranteed that Euclidean distances between any mandatory configurations is greater than 6.

**Task.**
Implement three functions in `solutions/broom_racing.py`:
- `gate_pass(start, goal) -> curve`: fly from `start` to `goal` (both `Configuration`).
- `catch_snitch(start, goal_xyz) -> curve`: reach `goal_xyz` (`XYZConfiguration`, only position matters).
- `catch_ball_and_gate(start, intermediate_goal, final_goal) -> curve`: reach the ball then fly through the gate.

Each returns a function `curve(s: ndarray) -> Configuration` with $s \in [0, 1]$. See `lib/broom_racing.py` for `Configuration`, `XYZConfiguration`, and constraint-checking helpers. Returned function will be tested to satisfy discretized EOM and curvature/pitch constraints. Total path length must be at most equal to reference implementation.

Visual explanation:

In [ ]:
from lib.broom_racing import show_quiddich_viewer

show_quiddich_viewer()

Examples of the opponent trajectories (cases correspond to those in open tests):

<img src="assets/quiddich_paths/opponent_gate_pass_0.png" alt="Opponent gate_pass (straight)" style="max-width: 420px;" />
<img src="assets/quiddich_paths/opponent_gate_pass_1.png" alt="Opponent gate_pass (curved)" style="max-width: 420px;" />
<img src="assets/quiddich_paths/opponent_gate_pass_2.png" alt="Opponent gate_pass (climb)" style="max-width: 420px;" />

<img src="assets/quiddich_paths/opponent_catch_snitch_0.png" alt="Opponent catch_snitch (horizontal)" style="max-width: 420px;" />
<img src="assets/quiddich_paths/opponent_catch_snitch_1.png" alt="Opponent catch_snitch (climb)" style="max-width: 420px;" />
<img src="assets/quiddich_paths/opponent_catch_snitch_2.png" alt="Opponent catch_snitch (height + lateral)" style="max-width: 420px;" />

<img src="assets/quiddich_paths/opponent_catch_ball_and_gate_0.png" alt="Opponent catch_ball_and_gate" style="max-width: 420px;" />
<img src="assets/quiddich_paths/opponent_catch_ball_and_gate_1.png" alt="Opponent catch_ball_and_gate (height)" style="max-width: 420px;" />

View your own trajectories:

In [ ]:
from lib.broom_racing import Configuration, XYZConfiguration, check_all, curve_length
from lib.broom_racing import show_broom_path
from solutions.broom_racing import gate_pass

start = Configuration(0, 0, 0, 0, 0)
goal = Configuration(8, 0, 0, 0, 0)
curve = gate_pass(start, goal)
show_broom_path(curve, start, goal, title="gate_pass example")
ok, errors = check_all(curve, start, goal=goal)
print(f"constraints ok: {ok}  length: {curve_length(curve):.2f}")
if errors:
    print("errors:", errors)

# Problem 3: Beads (10 pts + 5 bonus top-20 leaderboard)

**Trivia.** 
Imagine a situation where you have to squeeze a chain of beads in a very tight space with smallest footprint possible while satisfying an approximate non-penetration constraint.

**Model.**
We are considering a simplified model with each bead being a ball-socket joint, where we disregard axis-aligned rotation due to symmetry, as depicted in the picture:

<img src="assets/ball_socket_joint.jpg" alt="Ball Socket Joint" style="max-width: 100%;" />

Another simplification will be the assumption of infinitesimal widths of cylindrical part for the collision model. Ball parts are approximated as outer whole spheres which should not intersect in any valid configuration.

In [ ]:
from lib.beads import show_beads_viewer

link_lengths = np.array([4.0] * 20)
angles = np.array([[np.pi/6, np.pi/6]] * 19)
show_beads_viewer(link_lengths, angles)

**Task.**
Given a sequence of link lengths, provide an angle configuration that minimizes the bounding sphere radius of the chain under the constraint of non-penetration between spheres.
Test checks whether there are joint limit violations, self-intersections and achieved "compactness" is below that of a heuristic helical solution.

Implement `optimal_bead_config(link_lengths) -> angles` in `solutions/beads.py`.
- `link_lengths`: `(N,)` array of positive lengths each chosen randomly uniformly between 2 and 4; N links, N−1 universal joints, 2×(N−1) angles.
- **Spherical cap (joint limit):** each joint is parameterized by Euler angles $(\theta_1, \theta_2)$ (XYZ, third angle 0). The next link direction must lie in a spherical cap of angular radius $\pi/3$ around the previous link axis (identity direction). Equivalently, the great-circle angle from the identity satisfies $\cos\theta_1 \cos\theta_2 \geq \cos(\pi/3)$ (z-component of the rotated axis).
- Sphere radius = 1; ignore self-collision for consecutive links.

In [ ]:
from lib.beads import bounding_sphere_radius
from lib.beads import show_beads_viewer
from solutions.beads import optimal_bead_config

link_lengths = np.array([2.0] * 12)
angles = optimal_bead_config(link_lengths)
radius = bounding_sphere_radius(link_lengths, angles)
print(f"Bounding sphere radius: {radius:.4f}")
show_beads_viewer(link_lengths, angles)

Whatever you may think, the inspiration for the task came from this annoying thing which I always crave to wind up as tightly as possible.

<img src="assets/beads_inspiration.png" alt="Beaded string from office shades" style="max-width: 480px;" />